# 🥇 Gold Layer — Subscription Behaviour
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/silver/` and `medallion/gold/`, assembles a customer-level subscription profile covering order metrics, interval patterns, SKU/flavour variety, churn info, and conversion flags, and writes to `medallion/gold/`.

| Source Tables | Gold Table | Key Transformations |
|---|---|---|
| `silver_recharge_orders`, `silver_recharge_order_items`, `silver_recharge_recurring`, `silver_recharge_churned`, `silver_recharge_reactivated`, `gold_customer_orders` | `gold_subscription_behaviour` | Aggregate order metrics, compute interval stats & stretching flag, derive SKU/flavour variety, extract churn & reactivation info, flag Shopify conversion, assemble final profile |

## 0. Setup & Paths

In [18]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import glob
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. Subscription Behaviour (`silver_recharge_*` → `gold_subscription_behaviour`)

### Step 1 — Load silver recharge tables & aggregate order-level metrics per customer

In [19]:
# Load silver recharge tables
rc_orders    = pd.read_parquet(SILVER_DIR / 'silver_recharge_orders.parquet')
rc_checkout  = pd.read_parquet(SILVER_DIR / 'silver_recharge_order_items.parquet')
rc_recurring = pd.read_parquet(SILVER_DIR / 'silver_recharge_recurring.parquet')
rc_churned   = pd.read_parquet(SILVER_DIR / 'silver_recharge_churned.parquet')
rc_reactivated = pd.read_parquet(SILVER_DIR / 'silver_recharge_reactivated.parquet')

# Order-level metrics per customer
rc_orders['metric_date'] = pd.to_datetime(rc_orders['metric_date'], errors='coerce')
rc_orders = rc_orders.sort_values(['customer_id', 'metric_date'])

# Order counts by type
order_counts = rc_orders.groupby('customer_id').agg(
    total_rc_orders       =('recharge_order_id', 'count'),
    total_checkout_orders =('order_type', lambda x: (x == 'checkout').sum()),
    total_recurring_orders=('order_type', lambda x: (x == 'recurring').sum()),
    first_order_date      =('metric_date', 'min'),
    last_order_date       =('metric_date', 'max'),
    total_spend           =('order_total', 'sum'),
    avg_order_value       =('order_total', 'mean'),
    total_discounts       =('order_discounts', 'sum'),
).reset_index()

order_counts['total_spend']    = pd.to_numeric(order_counts['total_spend'], errors='coerce').round(2)
order_counts['avg_order_value']= pd.to_numeric(order_counts['avg_order_value'], errors='coerce').round(2)
order_counts['total_discounts']= pd.to_numeric(order_counts['total_discounts'], errors='coerce').round(2)

### Step 2 — Build Shopify <-> Recharge bridge & Compute order interval metrics (recurring orders only)

In [20]:
# ── Build Shopify <-> Recharge customer ID bridge ─────────────────────────────
raw_files = sorted([f for f in glob.glob(str(BASE / '1.customer_transaction' / '*.xlsx'))
                    if not f.split('/')[-1].startswith('~$')])
raw_orders = pd.concat([pd.read_excel(f, dtype=str) for f in raw_files], ignore_index=True)
top_rows = raw_orders[raw_orders['Top Row'].notna()][['ID', 'Customer: ID']].copy()
top_rows.columns = ['shopify_order_id', 'shopify_customer_id']

bridge = top_rows.merge(
    rc_orders[['shopify_order_id', 'customer_id']],
    on='shopify_order_id', how='inner'
)[['shopify_customer_id', 'customer_id']].drop_duplicates()

bridge.to_parquet(SILVER_DIR / 'silver_customer_id_bridge.parquet', index=False)
print(f"✅ Saved: silver_customer_id_bridge.parquet ({len(bridge):,} mappings)")

# Order interval metrics for Q9 
rc_orders['prev_order_date']    = rc_orders.groupby('customer_id')['metric_date'].shift(1)
rc_orders['order_interval_days']= (rc_orders['metric_date'] - rc_orders['prev_order_date']).dt.days

# Only use recurring orders for interval analysis
rc_recurring_only = rc_orders[rc_orders['order_type'] == 'recurring'].copy()
interval_stats = rc_recurring_only.groupby('customer_id').agg(
    avg_order_interval  =('order_interval_days', 'mean'),
    max_order_interval  =('order_interval_days', 'max'),
    std_order_interval  =('order_interval_days', 'std'),
    last_interval       =('order_interval_days', 'last'),
).reset_index()

interval_stats['avg_order_interval'] = interval_stats['avg_order_interval'].round(1)
interval_stats['max_order_interval'] = interval_stats['max_order_interval'].round(1)
interval_stats['std_order_interval'] = interval_stats['std_order_interval'].round(1)
interval_stats['last_interval']      = interval_stats['last_interval'].round(1)

# Interval stretching flag — last interval > 1.5x average (early churn warning)
interval_stats['is_interval_stretching'] = (
    interval_stats['last_interval'] > interval_stats['avg_order_interval'] * 1.5
)

✅ Saved: silver_customer_id_bridge.parquet (552 mappings)


### Step 3 — Derive SKU and flavour variety

In [21]:
# SKU and flavour variety for Q5 and Q6
all_items = pd.concat([rc_checkout, rc_recurring], ignore_index=True)
all_items = all_items[all_items['product_sku'].notna()].copy()

# Extract flavour from variant_title
def extract_flavour(title):
    if pd.isna(title): return None
    # Format: "1 x 1kg Pack / Thai Milk Tea" or "500g (20 serves) / Vanilla"
    if '/' in title:
        return title.split('/')[-1].strip()
    return None

all_items['flavour'] = all_items['variant_title'].apply(extract_flavour)

sku_variety = all_items.groupby('customer_id').agg(
    unique_skus          =('product_sku', 'nunique'),
    unique_flavours      =('flavour', 'nunique'),
    total_items_ordered  =('product_sku', 'count'),
    most_ordered_sku     =('product_sku', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    most_ordered_flavour =('flavour', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
).reset_index()

# Flavour rotation flag — ordered more than 1 unique flavour
sku_variety['is_flavour_rotator'] = sku_variety['unique_flavours'] > 1

### Step 4 — Extract subscription status, churn info & reactivation flag

In [22]:
# Subscription status and churn info 
# Churn — take worst churn per customer (if any active churns exist)
churn_info = rc_churned.groupby('customer_id').agg(
    churn_type          =('churn_type', 'first'),
    cancellation_reason =('cancellation_reason', lambda x: x.dropna().iloc[0] if x.notna().any() else None),
    subscription_activation_date=('subscription_activation_date', 'min'),
    subscription_churn_date     =('subscription_churn_date', 'max'),
    churned_skus        =('product_sku', lambda x: ', '.join(x.dropna().unique())),
).reset_index()

churn_info['subscription_activation_date'] = pd.to_datetime(
    churn_info['subscription_activation_date'], errors='coerce')
churn_info['subscription_churn_date'] = pd.to_datetime(
    churn_info['subscription_churn_date'], errors='coerce')
churn_info['subscription_lifetime_days'] = (
    churn_info['subscription_churn_date'] - churn_info['subscription_activation_date']
).dt.days

# Reactivated flag
reactivated_ids = set(rc_reactivated['customer_id'].dropna())

### Step 5 — Flag Shopify conversion (did recharge customer match a Shopify customer?)

In [23]:
# Conversion flag — did Shopify customer convert to subscriber? 
# Load gold_customer_orders to check if recharge customer_id matches shopify customer_id
co = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')
shopify_customers = set(co['customer_id'].dropna())
recharge_customers = set(rc_orders['customer_id'].dropna())
converted_customers = recharge_customers & shopify_customers

print(f"Recharge customers matched to Shopify: {len(converted_customers):,} / {len(recharge_customers):,}")

Recharge customers matched to Shopify: 0 / 575


### Step 6 — Assemble `gold_subscription_behaviour` & save

In [24]:
# Assemble gold_subscription_behaviour
sub = order_counts.copy()
sub = sub.merge(interval_stats, on='customer_id', how='left')
sub = sub.merge(sku_variety, on='customer_id', how='left')
sub = sub.merge(churn_info[['customer_id', 'churn_type', 'cancellation_reason',
                              'subscription_activation_date', 'subscription_churn_date',
                              'subscription_lifetime_days', 'churned_skus']],
                on='customer_id', how='left')

sub['is_churned']      = sub['churn_type'].notna()
sub['is_reactivated']  = sub['customer_id'].isin(reactivated_ids)
sub['is_converted']    = sub['customer_id'].isin(converted_customers)

# Subscription conversion — one-time buyers who later subscribed
# (checkout orders first, then recurring orders appear)
sub['converted_to_subscription'] = (
    (sub['total_checkout_orders'] > 0) & (sub['total_recurring_orders'] > 0)
)

gold_subscription_behaviour = sub

print(f"\ngold_subscription_behaviour: {gold_subscription_behaviour.shape[0]:,} rows × {gold_subscription_behaviour.shape[1]} cols")
print(f"\nChurned customers: {gold_subscription_behaviour['is_churned'].sum():,}")
print(f"Reactivated customers: {gold_subscription_behaviour['is_reactivated'].sum():,}")
print(f"Converted to subscription: {gold_subscription_behaviour['converted_to_subscription'].sum():,}")
print(f"Flavour rotators: {gold_subscription_behaviour['is_flavour_rotator'].sum():,}")
print(f"Interval stretching: {gold_subscription_behaviour['is_interval_stretching'].sum():,}")
print(f"\nCancellation reason breakdown:")
print(gold_subscription_behaviour['cancellation_reason'].value_counts(dropna=False).to_string())

gold_subscription_behaviour.to_parquet(GOLD_DIR / 'gold_subscription_behaviour.parquet', index=False)
print("\n✅ Saved: gold_subscription_behaviour.parquet")


gold_subscription_behaviour: 575 rows × 30 cols

Churned customers: 320
Reactivated customers: 44
Converted to subscription: 201
Flavour rotators: 155
Interval stretching: 14

Cancellation reason breakdown:
cancellation_reason
NaN                                      255
I already have more than I need           89
Other reason                              76
I no longer use this product              59
None                                      47
This was created by accident              26
This is too expensive                     14
I need it sooner                           5
I want a different product or variety      4

✅ Saved: gold_subscription_behaviour.parquet


#### Overall — 575 recharge customers
This is your entire subscriber base. A relatively small but high-value segment compared to the 13,885 total Shopify customers — these are the most engaged customers worth protecting.

#### Churned: 320 (55.7%)
Over half of all subscribers have churned — a very high churn rate. Combined with only 44 reactivated, most churned customers don't come back. This is the core problem Q5, Q6, and Q9 are trying to address.

#### Reactivated: 44 (7.7%)
Only 44 customers came back after churning. Small but worth studying for the reactivation fingerprint — what brought them back and what did they order on return.

#### Converted to subscription: 201 (35.0%)
201 customers started as one-time checkout buyers before converting to a recurring subscription. This is the key group for Q5 — what behavioural signals preceded their conversion?

#### Flavour rotators: 155 (27.0%)
27% of subscribers ordered more than one flavour across their lifetime. The remaining 73% stuck to a single flavour. For Q6 (Flavour Fatigue), the question is whether the 73% single-flavour customers churn faster than the 27% rotators.

#### Interval stretching: 14 (2.4%)
Only 14 customers show the early warning signal of their last order interval being 1.5x their average. Surprisingly low — this could mean most customers either churn abruptly rather than gradually stretching, or the 1.5x threshold is too conservative. Worth revisiting the threshold during EDA.

#### Cancellation reasons

NaN + None (302 combined, 52.5%) — no reason recorded, mostly passive churns (payment failures)

I already have more than I need (89, 15.5%) — customers stockpiling, reorder frequency too high

Other reason (76, 13.2%) — vague, not actionable

I no longer use this product (59, 10.3%) — product-market fit issue or lifestyle change

This was created by accident (26, 4.5%) — accidental subscriptions, worth flagging as a UX issue

This is too expensive (14, 2.4%) — price sensitivity, relevant for Q2 discount analysis

I need it sooner (5, 0.9%) — delivery frequency too slow

I want a different product or variety (4, 0.7%) — directly supports Q6, flavour fatigue signal